In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
train = pd.read_csv("./used_car_train_20200313.csv",sep = r'\s+')
test = pd.read_csv("./used_car_testA_20200313.csv", sep = r'\s+')
print("训练集大小", train.shape)
print("测试集大小", test.shape)

In [ ]:
### 查看原始数据
print("train:",train.head())
print("test:", test.head())
print("*" * 100)

### 缺失值
missing = train.isnull().sum()
print(missing)

print("*" * 100)

### 目标变量
train['price'].describe()
train['price'].hist(bins=50)


# 数据预处理

### 缺失值处理

In [ ]:
### notRepairedDamage属性中的缺失值以"-"表示，替换为nan
print(train['notRepairedDamage'].unique())
train['notRepairedDamage'] = train['notRepairedDamage'].replace('-', np.nan)
test['notRepairedDamage'] = test['notRepairedDamage'].replace('-', np.nan)

In [ ]:
### 提取目标列
y_train = train['price']
train.drop('price', axis=1, inplace=True)

### 统一处理，避免不一致
train['is_train'] = 1
test['is_train'] = 0
all_data = pd.concat([train, test], sort=False).reset_index(drop=True)

### 特征分类
# 真正的数值连续特征
num_cols = ['power', 'kilometer', 'v_0', 'v_1', 'v_2', 'v_3', 'v_4', 'v_5',
            'v_6', 'v_7', 'v_8', 'v_9', 'v_10', 'v_11', 'v_12', 'v_13', 'v_14']

# 本质是类别的特征
cat_cols = ['bodyType', 'fuelType', 'gearbox', 'notRepairedDamage', 
            'seller', 'offerType', 'brand', 'model', 'regionCode', 'name']

# 日期列
date_cols = ['regDate', 'creatDate']

### 类型转换
# 数值特征：强制转浮点型
for col in num_cols:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce').astype(float)

# 分类特征：强制转数值型
for col in cat_cols:
    all_data[col] = pd.to_numeric(all_data[col], errors='coerce')


### 缺失值处理
for col in all_data.columns:
    if col == 'is_train':
        continue
    if all_data[col].isnull().any():
        if col in num_cols:
            # 如果数值型特征为空，新建_ismissing特征列（是否为空本身可能也是一种信息），若为空则填1，反之填0
            all_data[col + '_ismissing'] = all_data[col].isnull().astype(int)
            all_data[col] = all_data[col].fillna(all_data[col].median())
        elif col in cat_cols:
            # 类别特征如果为空则填入-999，将其与正常类别区分
            all_data[col] = all_data[col].fillna(-999)

### 完成类型变量的转换
for col in cat_cols:
    all_data[col] = all_data[col].astype(int)

### 拆开
train = all_data[all_data['is_train'] == 1].drop('is_train', axis=1)
test  = all_data[all_data['is_train'] == 0].drop('is_train', axis=1)

### 加回price特征
train['price'] = y_train

### 日期处理

In [ ]:
for col in ['regDate', 'creatDate']:
    train[col] = pd.to_datetime(train[col], format='%Y%m%d', errors='coerce')
    test[col] = pd.to_datetime(test[col], format='%Y%m%d', errors='coerce')

### 构造特征：汽车使用天数
train['car_age_days'] = (train['creatDate'] - train['regDate']).dt.days
test['car_age_days'] = (test['creatDate'] - test['regDate']).dt.days


In [ ]:
print(train.head())

### 排除掉不符合题目要求的行 

In [ ]:
# 定义题目要求的合法取值
LEGAL_RULES = {
    "bodyType": [0,1,2,3,4,5,6,7],
    "fuelType": [0,1,2,3,4,5,6],
    "gearbox": [0,1],
    "notRepairedDamage": [0,1],
    "seller": [0,1],
    "offerType": [0,1],
}
POWER_RANGE = (0, 600)

### 清洗训练集
train = train[
    train["bodyType"].isin(LEGAL_RULES["bodyType"]) &
    train["fuelType"].isin(LEGAL_RULES["fuelType"]) &
    train["gearbox"].isin(LEGAL_RULES["gearbox"]) &
    train["notRepairedDamage"].isin(LEGAL_RULES["notRepairedDamage"]) &
    train["seller"].isin(LEGAL_RULES["seller"]) &
    train["offerType"].isin(LEGAL_RULES["seller"]) &
    train["power"].between(*POWER_RANGE)
].reset_index(drop=True)

### 清洗测试集
test["power"] = test["power"].clip(*POWER_RANGE)

### 结果一览
print("筛选约束完成！")
print("清洗后训练集形状：", train.shape)
print("清洗后测试集形状：", test.shape)